# Module 02 — CNNs Deep Dive (SOLUTIONS)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class DilatedBottleneck(nn.Module):
    """Bottleneck block with dilated 3x3 convolution."""
    expansion = 4

    def __init__(self, in_planes, planes, stride=1, dilation=2):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        # Dilated 3x3 conv — padding = dilation to preserve spatial size
        self.conv2 = nn.Conv2d(planes, planes, 3,
                               stride=stride, padding=dilation,
                               dilation=dilation, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, planes * self.expansion, 1, bias=False)
        self.bn3   = nn.BatchNorm2d(planes * self.expansion)

        self.shortcut = nn.Identity()
        if stride != 1 or in_planes != planes * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * self.expansion, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * self.expansion),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = F.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))
        return F.relu(out + self.shortcut(x))


block = DilatedBottleneck(256, 64, dilation=2)
x = torch.randn(2, 256, 28, 28)
out = block(x)
print('Output shape:', out.shape)  # (2, 256, 28, 28)